# Decentralizing Multilingual Language Technology
## A Preliminary Comparison of Cloud and Edge Inference Across Five Languages
**Author:** Pegah Merrikhi  
**Notebook:** Reproducible empirical benchmark analysis

### Abstract
This notebook reproduces a small paired benchmark comparing centralized cloud inference with decentralized edge inference for Persian, Azerbaijani, Turkish, Russian, and English. It reports latency/quality-style indicators (TTFT, TFF, LHI, PFS), paired descriptive and inferential statistics, typological summaries, exploratory associations, and regenerated figures. The unit of pairing is language: each language contributes one Cloud and one Edge observation (five pairs total).

### Research framing
The design contrasts a **centralized cloud** serving condition with **edge inference**, motivated by decentralization and multilingual access. The observations are a preliminary benchmark, not repeated trials or a representative sample of all speakers, models, devices, or networks. Consequently, p-values and correlations are descriptive/exploratory; no causal or broad population claim should be inferred from n=5 language pairs. The data workbook is the source of record; an embedded fallback reproduces its ten raw rows if the workbook is not available.

## 1. Environment setup and dependencies
Core requirements: Python, pandas, NumPy, SciPy, Matplotlib, and Seaborn. The plotting code uses standard Matplotlib/Seaborn APIs. No package installation is performed automatically; install missing dependencies in your environment before running.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", context="notebook")
SEED = 2025  # documents deterministic intent; no stochastic analysis is used
DATA_PATH = Path("/mnt/data/empirical_benchmark_results.xlsx")
OUT_DIR = Path("/mnt/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)
LANGUAGE_ORDER = ["Persian", "Azerbaijani", "Turkish", "Russian", "English"]
METRICS = ["TTFT_ms", "TFF", "LHI", "PFS"]

## 2. Raw benchmark data loading and verification
Load the `Benchmark Data` worksheet when available. It includes the exact ten language-condition records; summary rows, if any, are excluded by requiring `Condition` to be Cloud or Edge and `Language` to be one of the five named languages. If the workbook is unavailable, the exact ten observations are embedded below. The resulting paired table has one row per language and explicit `_Cloud` / `_Edge` columns.

In [ ]:
fallback_rows = [
    ("Persian", "Non-agglutinative", "Cloud", 310, 2.1, 0.72, 3.2),
    ("Persian", "Non-agglutinative", "Edge",   95, 1.4, 0.31, 4.5),
    ("Azerbaijani", "Agglutinative", "Cloud", 480, 3.8, 0.81, 2.8),
    ("Azerbaijani", "Agglutinative", "Edge",  142, 1.9, 0.38, 4.2),
    ("Turkish", "Agglutinative", "Cloud", 510, 3.5, 0.78, 3.0),
    ("Turkish", "Agglutinative", "Edge",   128, 1.7, 0.34, 4.4),
    ("Russian", "Non-agglutinative", "Cloud", 400, 2.4, 0.65, 3.5),
    ("Russian", "Non-agglutinative", "Edge",  105, 1.5, 0.39, 4.1),
    ("English", "Non-agglutinative", "Cloud", 460, 1.1, 0.21, 4.8),
    ("English", "Non-agglutinative", "Edge",   121, 1.1, 0.25, 4.6),
]
columns = ["Language", "Language_Type", "Condition", *METRICS]
if DATA_PATH.exists():
    raw = pd.read_excel(DATA_PATH, sheet_name="Benchmark Data")
    source = f"Excel workbook: {DATA_PATH}"
else:
    raw = pd.DataFrame(fallback_rows, columns=columns)
    source = "embedded exact-observation fallback"

raw = raw.loc[raw["Language"].isin(LANGUAGE_ORDER) & raw["Condition"].isin(["Cloud", "Edge"]), columns].copy()
raw["Language"] = pd.Categorical(raw["Language"], categories=LANGUAGE_ORDER, ordered=True)
raw = raw.sort_values(["Language", "Condition"]).reset_index(drop=True)
assert len(raw) == 10, f"Expected exactly 10 raw records, got {len(raw)}"
assert not raw.duplicated(["Language", "Condition"]).any(), "Duplicate language-condition record"
assert raw.groupby("Language", observed=False)["Condition"].nunique().eq(2).all(), "Every language must have both conditions"
assert raw[METRICS].notna().all().all(), "Missing metric value(s)"
print("Source:", source)
print("Raw data shape:", raw.shape)
display(raw)

In [ ]:
# Wide paired data: one row per language, with explicit condition-specific measures
paired = raw.pivot(index=["Language", "Language_Type"], columns="Condition", values=METRICS)
paired.columns = [f"{metric}_{condition}" for metric, condition in paired.columns]
paired = paired.reset_index()
paired["Language"] = pd.Categorical(paired["Language"], categories=LANGUAGE_ORDER, ordered=True)
paired = paired.sort_values("Language").reset_index(drop=True)
for metric in METRICS:
    paired[f"{metric}_Diff_Cloud_minus_Edge"] = paired[f"{metric}_Cloud"] - paired[f"{metric}_Edge"]
assert len(paired) == 5 and paired.filter(like="_Diff_").shape[1] == 4
print("Paired table shape:", paired.shape)
display(paired)

## 3. Descriptive statistics and paired comparisons
For each metric, the difference is defined as **Cloud − Edge**. Report condition means and sample SDs (`ddof=1`), the mean paired difference, paired t statistic (df=4), two-sided p-value, Cohen's *d*<sub>z</sub> = mean difference / SD of paired differences, and Wilcoxon signed-rank results. Bonferroni adjustment covers the four paired t-tests (family-wise α=.05; adjusted p = min(4p, 1)). The t-test assumes independent language-level pairs and approximately normal pair differences—assumptions that cannot be meaningfully assessed with five languages.

### Wilcoxon handling note (especially TFF)
TFF has four positive differences and an exact zero for English. With the zero excluded, only four nonzero pairs remain; the exact two-sided signed-rank p-value is **0.125**. A value of **0.0625** results from treating all five observations as nonzero and all five ranks as positive, which is not an appropriate exact calculation when one observed difference is zero. SciPy's default zero handling drops the zero, but switches to an asymptotic approximation in this tied/zero situation; it reproduces the workbook's **0.067889** (no continuity correction). These numbers answer different computational conventions and should not be presented as interchangeable exact and asymptotic estimates. For transparency, this notebook outputs all three values. For other metrics, SciPy's automatic method is recorded alongside exact calculation after zero removal.

In [ ]:
rows = []
for metric in METRICS:
    cloud = paired[f"{metric}_Cloud"].astype(float).to_numpy()
    edge = paired[f"{metric}_Edge"].astype(float).to_numpy()
    diff = cloud - edge
    t_res = stats.ttest_rel(cloud, edge, alternative="two-sided")
    # Explicit exact signed-rank test on nonzero differences; ties are handled by SciPy's exact distribution.
    nonzero = diff[diff != 0]
    try:
        w_exact = stats.wilcoxon(nonzero, np.zeros_like(nonzero), alternative="two-sided", method="exact")
        exact_p = float(w_exact.pvalue)
        w_stat_exact = float(w_exact.statistic)
    except ValueError:
        exact_p, w_stat_exact = np.nan, np.nan
    # Reproduce SciPy's default auto choice. In TFF, zero is removed and normal approximation is used.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        w_auto = stats.wilcoxon(cloud, edge, alternative="two-sided", method="auto", zero_method="wilcox")
    # Show the naive five-pair sign-rank result requested for contrast (NOT a valid zero-aware exact result).
    naive_p = np.nan
    if metric == "TFF":
        naive_p = 0.0625  # two-sided all-positive exact signed ranks under the false assumption of 5 nonzero pairs
    rows.append({
        "Metric": metric,
        "Cloud_Mean": cloud.mean(), "Cloud_SD": cloud.std(ddof=1),
        "Edge_Mean": edge.mean(), "Edge_SD": edge.std(ddof=1),
        "Mean_Diff_Cloud_minus_Edge": diff.mean(),
        "Cohens_dz": diff.mean() / diff.std(ddof=1),
        "t_statistic": t_res.statistic, "df": len(diff)-1, "p_ttest": t_res.pvalue,
        "p_ttest_bonferroni_4": min(1.0, t_res.pvalue * len(METRICS)),
        "Wilcoxon_W_auto": w_auto.statistic, "Wilcoxon_p_auto": w_auto.pvalue,
        "Wilcoxon_p_exact_nonzero": exact_p,
        "TFF_naive_p_exact_n5_invalid_if_zero": naive_p,
        "n_nonzero_pairs": len(nonzero),
    })
summary = pd.DataFrame(rows)
display(summary.round(6))

# Sanity checks against the source workbook's reported paired means and t-test p-values.
if DATA_PATH.exists():
    reported = pd.read_excel(DATA_PATH, sheet_name="Statistical Summary").set_index("Metric")
    for metric in METRICS:
        assert np.isclose(summary.set_index("Metric").loc[metric, "Cloud_Mean"], reported.loc[metric, "Cloud_Mean"])
        assert np.isclose(summary.set_index("Metric").loc[metric, "Edge_Mean"], reported.loc[metric, "Edge_Mean"])
        assert np.isclose(summary.set_index("Metric").loc[metric, "p_ttest"], reported.loc[metric, "p_ttest"], rtol=1e-10)
    print("Verified: Cloud/Edge means and paired t-test p-values match workbook Statistical Summary.")
print("TFF exact zero-excluded p:", summary.set_index("Metric").loc["TFF", "Wilcoxon_p_exact_nonzero"])
print("TFF naive n=5 p (invalid with zero):", summary.set_index("Metric").loc["TFF", "TFF_naive_p_exact_n5_invalid_if_zero"])
print("TFF SciPy auto p:", summary.set_index("Metric").loc["TFF", "Wilcoxon_p_auto"])

## 4. Typological breakdown: agglutinative vs non-agglutinative
The source classifies Azerbaijani and Turkish as agglutinative; Persian, Russian, and English are grouped as non-agglutinative for this benchmark. For each metric and condition, calculate the mean across languages within each group, along with group size. Also compute language-level Cloud−Edge differences by typology. These are **descriptive subgroup summaries**; group sizes are 2 versus 3, the classification is coarse, and languages are not interchangeable independent replications of typology.

In [ ]:
typology_means = (raw.groupby(["Language_Type", "Condition"], observed=True)[METRICS]
                  .agg(["mean", "std", "count"]).reset_index())
# Flatten multi-index columns for usable CSV/table output
typology_means.columns = ["_".join(str(x) for x in col if str(x) != "") if isinstance(col, tuple) else col for col in typology_means.columns]
display(typology_means.round(4))

diff_long = paired.melt(id_vars=["Language", "Language_Type"],
                        value_vars=[f"{m}_Diff_Cloud_minus_Edge" for m in METRICS],
                        var_name="Metric", value_name="Cloud_minus_Edge")
diff_long["Metric"] = diff_long["Metric"].str.replace("_Diff_Cloud_minus_Edge", "", regex=False)
typology_diffs = diff_long.groupby(["Language_Type", "Metric"], observed=True)["Cloud_minus_Edge"].agg(["mean", "std", "count"]).reset_index()
display(typology_diffs.round(4))

## 5. Exploratory bivariate analysis and correlation caveats
Plot LHI versus PFS for each language-condition pair and mark condition centroids. Report pooled correlation across ten rows and within-condition correlations (five rows each), as well as the across-language correlations of condition means. Interpret these only as exploratory. Pooling conditions can create a correlation driven largely by the Cloud–Edge separation rather than a within-condition relationship—an aggregation/Simpson-type issue. Treating pooled rows as independent also ignores repeated language pairs. With only five language pairs (`n=5` within each condition), correlations are highly unstable, have very low power, and cannot establish ecological or individual-level relationships. Correlation does not imply causation.

In [ ]:
pooled = raw.melt(id_vars=["Language", "Language_Type", "Condition"], value_vars=["LHI", "PFS"], var_name="Measure", value_name="Value")
# Correlations are calculated on the raw wide columns, not the plotting long table.
def corr_report(frame, label):
    r, p = stats.pearsonr(frame["LHI"].astype(float), frame["PFS"].astype(float))
    return {"Scope": label, "n": len(frame), "Pearson_r": r, "p_value_unadjusted": p}
correlations = [corr_report(raw, "Pooled 10 language-condition records")]
for condition, sub in raw.groupby("Condition"):
    correlations.append(corr_report(sub, f"{condition} only (5 languages)"))
condition_means = raw.groupby("Condition")[["LHI", "PFS"]].mean().reset_index()
correlations.append(corr_report(condition_means, "Two condition centroids (n=2; illustrative only)"))
correlation_table = pd.DataFrame(correlations)
display(correlation_table.round(4))
print("Caution: pooled observations repeat the same five languages across conditions; pooled r is not an independent-row estimate.")

## 6. Figure regeneration and visualization
Figures show the five-language benchmark only. Bars display condition means; error bars are ±1 sample SD across languages (descriptive dispersion, **not** confidence intervals). Scatter points are individual language-condition observations and larger symbols show condition centroids. The typological plots display subgroup means and SD; the small n per category makes visual comparisons uncertain.

In [ ]:
# Paired metric overview: TTFT and TFF, mean +/- sample SD
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, metric, title in zip(axes, ["TTFT_ms", "TFF"], ["Time to first token (ms)", "TFF"]):
    means = [paired[f"{metric}_{c}"].mean() for c in ["Cloud", "Edge"]]
    sds = [paired[f"{metric}_{c}"].std(ddof=1) for c in ["Cloud", "Edge"]]
    ax.bar(["Cloud", "Edge"], means, yerr=sds, capsize=5, color=["#4472C4", "#70AD47"], alpha=.9)
    ax.set_title(title); ax.set_ylabel("Mean +/- SD"); ax.set_xlabel("Inference condition")
    for i, val in enumerate(means): ax.text(i, val + sds[i] + max(means)*.025, f"{val:.2f}", ha="center", fontsize=9)
fig.suptitle("Cloud vs Edge benchmark (n=5 paired languages)")
fig.tight_layout()
plt.show()

In [ ]:
# Exploratory LHI vs PFS scatter with language labels and condition centroids
fig, ax = plt.subplots(figsize=(7.5, 5.5))
palette = {"Cloud": "#4472C4", "Edge": "#70AD47"}
for condition, sub in raw.groupby("Condition"):
    ax.scatter(sub["LHI"], sub["PFS"], label=condition, s=75, color=palette[condition], alpha=.8)
    for _, r in sub.iterrows(): ax.annotate(str(r["Language"]), (r["LHI"], r["PFS"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
centroids = raw.groupby("Condition")[["LHI", "PFS"]].mean()
for condition, r in centroids.iterrows():
    ax.scatter(r["LHI"], r["PFS"], marker="X", s=220, color=palette[condition], edgecolor="black", linewidth=.8, zorder=5)
    ax.annotate(f"{condition} centroid", (r["LHI"], r["PFS"]), xytext=(7, -13), textcoords="offset points", weight="bold", fontsize=9)
ax.set(title="LHI vs PFS by condition (exploratory)", xlabel="LHI", ylabel="PFS")
ax.legend(title="Condition")
fig.tight_layout(); plt.show()

In [ ]:
# Typological subgroup means, separately by condition; display all four metrics
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, metric in zip(axes.flat, METRICS):
    sns.barplot(data=raw, x="Language_Type", y=metric, hue="Condition", order=["Agglutinative", "Non-agglutinative"],
                errorbar="sd", capsize=.08, palette=palette, ax=ax)
    ax.set_title(metric); ax.set_xlabel(""); ax.set_ylabel("Mean +/- SD across languages")
    ax.tick_params(axis="x", rotation=12)
    if ax is not axes.flat[0] and ax.legend_: ax.legend_.remove()
axes.flat[0].legend(title="Condition")
fig.suptitle("Typological breakdown (group means; very small groups)", y=1.02)
fig.tight_layout(); plt.show()

## 7. Conclusion, methodological safeguards, and exports
The raw paired results show lower mean TTFT at the edge and condition differences in the other benchmark measures, but the evidence is preliminary. Statistical significance is reported unadjusted and with Bonferroni adjustment; multiplicity correction makes clear that the apparent findings are fragile at this scale. No language-level repeated-run data, device/network metadata, uncertainty model, or population sampling is available in this spreadsheet.

Safeguards: (1) preserve language pairing; (2) identify Cloud−Edge direction explicitly; (3) report effect sizes and raw-scale summaries; (4) distinguish exact Wilcoxon from zero-handled asymptotic results; (5) label subgroup and correlation analyses exploratory; (6) avoid causal, universal typological, or individual-level conclusions; (7) use the five languages, not ten condition rows, as the independent paired units.

The final cell writes reproducibility CSVs to `/mnt/data`: clean paired observations, paired-statistics summary, typological condition summaries, typological paired-difference summaries, and correlation summaries.

In [ ]:
# Export tidy tables. CSVs are supplementary; rerunning this cell overwrites these generated outputs.
exports = {
    "ssm_edge_clean_paired_data.csv": paired,
    "ssm_edge_paired_statistics.csv": summary,
    "ssm_edge_typology_condition_summary.csv": typology_means,
    "ssm_edge_typology_paired_differences.csv": typology_diffs,
    "ssm_edge_correlation_summary.csv": correlation_table,
}
for filename, frame in exports.items():
    target = OUT_DIR / filename
    frame.to_csv(target, index=False)
    print(f"Wrote {target} ({len(frame)} rows)")

# Final reproducibility checks
assert summary["df"].eq(4).all()
assert np.isclose(summary.set_index("Metric").loc["TTFT_ms", "Mean_Diff_Cloud_minus_Edge"], 313.8)
assert np.isclose(summary.set_index("Metric").loc["TFF", "Mean_Diff_Cloud_minus_Edge"], 1.06)
print("All final assertions passed. Notebook analysis is complete.")